![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, watsonx.data, LangChain, and vector indexes to chat with a document (RAG)

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook demonstrates how to reproduce the behaviour of chat with a document and vector indexes programmatically through watsonx APIs and clients.

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Learning goal

The purpose of this notebook is to replicate chat with a documents behavior programmatically by integrating document and vector indexes with watsonx APIs and watsonx.data Milvus.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Build Vector Index](#Build-Vector-Index)
3. [Deploy AI Service](#Deploy-AI-Service)
4. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact with your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
Install the required libraries for this notebook, ensuring that in an airgap environment the local PyPI repository is populated with these dependencies.

**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U "ibm-watsonx-ai[rag]" | tail -n 1
%pip install -U jq | tail -n 1
%pip install -U docx2txt | tail -n 1
%pip install -U tiktoken | tail -n 1
%pip install -U unstructured | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

### Working with projects

First of all, you need to create a project that will be used for your work. If you do not have a project created already, follow the steps below:

- Open IBM Cloud Pak® main page
- Click all projects
- Create an empty project
- Copy `project_id` from url and paste it below

**Action**: Assign project ID below

In [4]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials, project_id)

<a id="Build-Vector-Index"></a>
## Build Vector Index
The following code demonstrates how to generate embeddings from a list of data assets within a Milvus collection and save them as a vector index asset.

### Collect data assets and connections to use in the vector index

Retrieve a list of data assets available in the project

In [6]:
client.data_assets.list()

,NAME,ASSET_TYPE,SIZE,ASSET_ID
0,state_of_the_union.txt,data_asset,39027,019e2b4b-0fb3-72af-8cb2-cdf28bd66307


Specify the data assets to be incorporated into the vector index.

In [7]:
data_assets = ["PASTE_YOUR_DATA_ASSET_ID_HERE"]

Retrieve a list of connections available in the project

In [8]:
client.connections.list()

,NAME,ID,CREATED,DATASOURCE_TYPE_ID
0,Milvus,019dcf2a-01bf-723f-9d37-6a5d5c7728ed,2026-04-27T13:38:49Z,da6d9a4e-992e-4965-b9b2-e9db0e76cd0c


Configure the properties of the watsonx.data Milvus instance and collection by specifying the details of the connection.

In [9]:
connection_id = "PASTE_YOUR_CONNECTION_ID_HERE"
collection = input("Enter your collection name:")
database = "default"

#### Create vector index

**Note**: This vector index does not require patching when updates are made to the collection.

Configure the vector index settings.

In [10]:
embedding_model_id = client.foundation_models.EmbeddingModels.SLATE_125M_ENGLISH_RTRVR
vector_index_name = input("Enter vector index name:")
chunk_size, chunk_overlap = 2000, 200

Utilizing the settings defined above, we construct the payload for the vector index, consolidating all configuration parameters into a structured format for subsequent processing.

In [11]:
from ibm_watsonx_ai.foundation_models.utils import VectorIndexes

vector_indexes = VectorIndexes(client)
vector_index_details = vector_indexes.create(
    name=vector_index_name,
    data_assets=data_assets,
    store={
        "type": "watsonx.data",
        "connection_id": connection_id,
        "index": collection,
        "database": database,
    },
    settings={
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "split_pdf_pages": True,
        "top_k": 5,
        "rerank": False,
        "embedding_model_id": embedding_model_id,
        "schema_fields": {
            "document_name": "document_name",
            "text": "text",
            "page_number": "page",
        },
    },
    status="ready",
)

In [12]:
# The following schema delineates the documents being created within the Milvus collection.
vector_store_schema = vector_index_details["settings"]["schema_fields"]
text_field = vector_store_schema.get("text")

### Initialize vector store

Instantiate the `VectorStore` class for the watsonx.data Milvus collection. By default, `VectorStore` will create a collection with the nama as is stated in `collection` variable. If collection with that name already exist, no new collection is created and running method `VectorStore.add_documents` the new documents will be added to the existing collection. To drop old collection and create a new one with the same name, please set `drop_old` param to `True` in `VectorStore` constructor.

In [13]:
from ibm_watsonx_ai.foundation_models import Embeddings
from ibm_watsonx_ai.foundation_models.extensions.rag.vector_stores import VectorStore

embeddings = Embeddings(
    model_id=embedding_model_id,
    api_client=client,
    params={"truncate_input_tokens": 512},
)

vector_store = VectorStore(
    api_client=client,
    connection_id=connection_id,
    embeddings=embeddings,
    index_name=collection,
    database=database,
    consistency_level="Strong",
    connection_args={"secure": True},
    text_field=text_field,
)

### Data Asset Processing

The following cells manage the parsing of data assets, their conversion into LangChain documents, and the generation of embeddings in the Milvus collection.

Define the document loader to be used for each file type

In [14]:
from langchain.document_loaders import (
    CSVLoader,
    Docx2txtLoader,
    JSONLoader,
    PyPDFLoader,
    TextLoader,
    UnstructuredExcelLoader,
    UnstructuredHTMLLoader,
    UnstructuredMarkdownLoader,
    UnstructuredPowerPointLoader,
)

mime_type_mappings = {
    "text/plain": TextLoader,
    "application/pdf": PyPDFLoader,
    "text/csv": CSVLoader,
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": Docx2txtLoader,
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": UnstructuredExcelLoader,
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": UnstructuredPowerPointLoader,
    "text/markdown": UnstructuredMarkdownLoader,
    "application/json": JSONLoader,
    "text/html": UnstructuredHTMLLoader,
}

Initialize the text splitter used to partition documents into manageable chunks.

In [15]:
from ibm_watsonx_ai.foundation_models.extensions.rag.chunker import LangChainChunker

text_splitter = LangChainChunker(
    method="recursive", chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

Define a function to parse the raw content of a data asset.

In [16]:
def load_document(data_asset: dict) -> list:
    """
    Load and split the document from a data asset.

    Args:
        data_asset (dict): A dictionary containing metadata and entity details of the data asset.

    Returns:
        list: The loaded and split content of the document.
    """
    asset_id = data_asset["metadata"]["asset_id"]
    filename = data_asset["metadata"]["name"]
    file_path = client.data_assets.download(asset_id, filename)
    mime_type = data_asset["entity"]["data_asset"]["mime_type"]

    # Get the correct loader for the MIME type.
    if mime_type == "application/json":
        loader = mime_type_mappings[mime_type](
            filename, jq_schema=".", text_content=False
        )
    else:
        loader = mime_type_mappings[mime_type](file_path)
    return loader.load_and_split()

Define a function to incorporate additional metadata into a LangChain document, such as the document name and an optional page number.

In [17]:
def compute_documents_metadata(document_name: str, loaded_documents: list) -> list:
    """
    Compute and update metadata for a list of loaded documents.

    Args:
        document_name (str): The name to be assigned to each document.
        loaded_documents (list): A list of documents from which metadata is extracted and updated.

    Returns:
        list: A list of documents with enriched metadata.
    """
    filtered_documents = []
    for document in loaded_documents:
        computed_document_data = {
            "metadata": {"source": document.model_dump()["metadata"]["source"]}
        }
        computed_document_data["metadata"][
            vector_store_schema.get("page_number")
        ] = document.model_dump()["metadata"].get("page", 0)
        computed_document_data["metadata"][
            vector_store_schema.get("document_name")
        ] = document_name
        filtered_documents.append(document.model_copy(update=computed_document_data))

    return filtered_documents

Define a function to extract and segment document chunks from an individual data asset.

In [18]:
def process_document(data_asset: dict) -> list:
    """
    Process a single data asset by loading, enriching, and splitting its content into document chunks.

    Args:
        data_asset (dict): A dictionary representing the data asset, which contains metadata and other details.

    Returns:
        list: A list of document chunks obtained after splitting the enriched document.
    """
    print("Processing", data_asset["metadata"]["name"])
    loaded_documents = load_document(data_asset)
    filtered_documents = compute_documents_metadata(
        data_asset["metadata"]["name"], loaded_documents
    )

    return text_splitter.split_documents(filtered_documents)

Define a function that retrieves document chunks for all available data assets.

In [19]:
def process_documents(data_asset_ids: list) -> list:
    """
    Process and retrieve documents from the given list of data asset IDs.

    Parameters:
        data_asset_ids (list): A list of data asset identifiers to be processed.

    Returns:
        list: A list of processed documents aggregated from the provided data assets.
    """
    documents = []
    for data_asset_id in data_asset_ids:
        data_asset = client.data_assets.get_details(data_asset_id)

        if data_asset["metadata"]["asset_type"] == "data_asset":
            document = process_document(data_asset)
            documents += document

    return documents

Process the data assets.

In [20]:
documents = process_documents(data_assets)

Processing state_of_the_union.txt
Successfully saved data asset content to file: 'state_of_the_union.txt'


### Create embeddings

Generate embeddings for the document chunks and add them into the Milvus collection.

In [21]:
vector_store.add_documents(content=documents, batch_size=200)

['24bacc4d64f5d0c78783dcbf79809cd1655b973c9388f9493218d9e4632fbf39',
 '08acc0a92ba5908bbf209f3a1d152a0201a34bbf6134c2c75455956afab98662',
 '2e603596ece7279a72b0632a042b8eb5ca043f4f39e3d0ca590f2ce86d2c876a',
 '72b96c2f6cb356e497660e6f85d9286b7eea134624b8fe211630fd369d38151c',
 'a19fd6e12ebfa25e5065f2b004c68850efaafc9b061111f8ca6013d7cfa2f29b',
 'ec6b5f72072b2f51de7c23ba6be83b09f27fcb893f55fb51f3ba857cdf62e73b',
 '41d59f011da5fdd2c0e9d644e6390916047c7f3a5153fb29a666b6460da5fffb',
 'a4340b2c471b3b38b08611fe9e66a8c81847ee1b1998c3da1a9ac3e6c5cae8ed',
 '76c8ae493e7b64c925ba37291715b1a2d993e64150f39d30781844e01692e505',
 'a0280493cb9f3948d9976b5b7235c0f54633f3d31964750ec0f6c9ba015359c3',
 'eedc69a8d922ae50d1ae0d69075f22537fb4a7b603f79deb51d6dba64a9ae01f',
 'f1da3ddd402de0dc9b2a26d05508e415fb1f5130ab17e34ccdd9a7a9229733d4',
 '40c78fbc0e7414c674b85084f6999eae1b162507c0d2fa5f608e33ab4f4c0d07',
 'e8d530c31f357eccd0730a3439e3e11d677df5b2566e988f74d39c7004a55f2b',
 '49fecf815dad190dfe84ec862577482e

<a id="Deploy-AI-Service"></a>
## Deploy AI Service

Below is a step-by-step example demonstrating how to deploy a prompt that interacts with a document using vector indexes.

*Note*: This feature is available only for models supported by the Chat AP

You can use the `list` method to display all existing spaces.

In [ ]:
client.spaces.list()

Define the space for deploying the AI service.

In [22]:
space_id = "ENTER YOUR SPACE ID HERE"

client.set.default_space(space_id)

Unsetting the project_id ...


'SUCCESS'

Promote the vector index to the designated space.

In [ ]:
vector_index_id = client.spaces.promote(
    vector_index_details["id"], project_id, space_id
)

### AI Service function definition
Define the AI service function to handle data retrieval and inference.

In [24]:
params = {
    "space_id": space_id,
    "vector_index_id": vector_index_id,
    "url": credentials.url,
}


def gen_ai_service(context, params=params):
    # import dependencies
    from ibm_watsonx_ai.client import APIClient, Credentials
    from ibm_watsonx_ai.foundation_models import Embeddings, ModelInference, Rerank
    from ibm_watsonx_ai.foundation_models.extensions.rag import Retriever
    from ibm_watsonx_ai.foundation_models.extensions.rag.vector_stores import (
        VectorStore,
    )

    vector_index_id = params.get("vector_index_id")
    space_id = params.get("space_id")
    url = params.get("url")

    # Inference details
    system_prompt = "You are a cautious assistant. You carefully follow instructions. You are helpful and harmless and you follow ethical guidelines and promote positive behavior. You are a AI language model designed to function as a specialized Retrieval Augmented Generation (RAG) assistant. When generating responses, prioritize correctness, i.e., ensure that your response is correct given the context and user query, and that it is grounded in the context. Furthermore, make sure that the response is supported by the given document or context. Always make sure that your response is relevant to the question. If an explanation is needed, first provide the explanation or reasoning, and then give the final answer. Avoid repeating information unless asked."
    inference_params = {"max_tokens": 2000, "temperature": 0}

    # Setup client
    credentials = Credentials(
        url=url,
        token=context.generate_token(),
        instance_id="openshift",
        version="5.4",
    )

    client = APIClient(credentials, space_id=space_id)

    # Get vector index details
    vector_index_details = client.data_assets.get_details(vector_index_id)
    vector_index_properties = vector_index_details["entity"]["vector_index"]

    def rerank(inner_client, documents, query, top_n):
        """
        Rerank a list of documents based on a query using a cross-encoder model.

        Parameters:
            inner_client: An API client instance used to interact with the underlying service.
            documents (list): A list of documents to be reranked.
            query (str): The query string used to evaluate the relevance of each document.
            top_n (int): The number of top documents to return after reranking.

        Returns:
            list: A new list of documents ordered by their relevance to the query.
        """
        reranker = Rerank(
            model_id="cross-encoder/ms-marco-minilm-l-12-v2",
            api_client=inner_client,
            params={"return_options": {"top_n": top_n}, "truncate_input_tokens": 512},
        )

        reranked_results = reranker.generate(query=query, inputs=documents)["results"]

        new_documents = [documents[result["index"]] for result in reranked_results]

        return new_documents

    def format_messages(messages, documents, system_prompt):
        """
        Format conversation messages by appending contextual information and prepending a system prompt.

        Parameters:
            messages (list): A list of message dictionaries, where each dictionary must include a "content" key.
            documents (list): A list of document strings that will be combined to form context.
            system_prompt (str): The system prompt to be inserted as the first message in the conversation.

        Returns:
            list: The updated list of messages including the reformatted last message and the prepended system message.
        """
        context = "\n".join(documents)

        # Append context to the last message.
        if messages:
            content = messages[-1].get("content", "")
            # Format of this string may be model dependent
            messages[-1]["content"] = (
                f"Use the following pieces of context to answer the question.\n\n"
                f"{context}\n\n"
                f"Question: {content}\n"
            )

        # Prepend the system prompt.
        messages.insert(0, {"role": "system", "content": system_prompt})

        return messages

    def inference_model(inference_model_id, inner_client, messages, stream):
        """
        Retrieve document chunks, incorporate contextual information, and generate a grounded response.

        Parameters:
            inference_model_id: ID of model that will be used for inferencing
            inner_client: An API client instance used for connecting to the vector store and the inference model.
            messages (list): A list of message dictionaries representing the conversation history. The content of
                            the last message is used as the query for document retrieval.
            stream (bool): If True, the inference model returns a streaming response; otherwise, it returns a complete response.

        Returns:
            The generated response from the inference model, either as a stream or as a complete message.
        """
        emb = Embeddings(
            model_id=vector_index_properties["settings"]["embedding_model_id"],
            api_client=inner_client,
            params={"truncate_input_tokens": 512},
        )

        top_n = (
            20
            if vector_index_properties["settings"].get("rerank")
            else int(vector_index_properties["settings"]["top_k"])
        )

        vector_store = VectorStore(
            client=inner_client,
            connection_id=vector_index_properties["store"]["connection_id"],
            embeddings=emb,
            index_name=vector_index_properties["store"]["index"],
            database=vector_index_properties["store"]["database"],
            consistency_level="Strong",
            connection_args={"secure": True},
            text_field=vector_index_properties["settings"]["schema_fields"]["text"],
            search_params={
                "ef": 2 * top_n
            },  # `ef` param needs to be larger than `top_n` param
        )

        # Retrieve document chunks from the vector index
        query = messages[-1].get("content")

        retriever = Retriever(vector_store=vector_store, number_of_chunks=top_n)
        documents = retriever.retrieve(query)

        def get_doc_content(doc):
            """
            Extract the content from a document.

            Parameters:
                doc: A document object with a 'page_content' attribute.

            Returns:
                str: The textual content of the document.
            """
            return doc.page_content

        document_contents = list(map(get_doc_content, documents))

        # Use reranking if enabled
        if vector_index_properties["settings"].get("rerank"):
            document_contents = rerank(
                inner_client,
                document_contents,
                query,
                vector_index_properties["settings"]["top_k"],
            )

        # Generate grounded response using the inference details
        messages = format_messages(
            messages, document_contents, system_prompt=system_prompt
        )

        model = ModelInference(
            model_id=inference_model_id,
            params=inference_params,
            api_client=inner_client,
            space_id=space_id,
        )

        if stream is True:
            generated_response = model.chat_stream(messages=messages)
        else:
            generated_response = model.chat(messages=messages)

        return generated_response

    def get_inner_client(context):
        """
        Set up and return an inner API client using the provided context.

        Parameters:
            context: An object used in AI services deployment runtime

        Returns:
            APIClient: An instance of APIClient configured with the constructed credentials and space ID.
        """
        inner_credentials = Credentials(
            url=url,
            token=context.get_token(),
            instance_id="openshift",
            version="5.4",
        )
        inner_client = APIClient(inner_credentials, space_id=space_id)
        return inner_client

    def generate(context):
        payload = context.get_json()
        messages = payload.get("messages")
        inference_model_id = payload.get("model_id")
        inner_client = get_inner_client(context)

        results = inference_model(inference_model_id, inner_client, messages, False)

        response = {"headers": {"Content-Type": "application/json"}, "body": results}

        return response

    def generate_stream(context):
        payload = context.get_json()
        messages = payload.get("messages")
        inference_model_id = payload.get("model_id")
        inner_client = get_inner_client(context)
        response_stream = inference_model(
            inference_model_id, inner_client, messages, True
        )

        yield from response_stream

    return generate, generate_stream

Define the request and response schemas for the AI service.

In [25]:
request_schema = {
    "application/json": {
        "$schema": "http://json-schema.org/draft-07/schema#",
        "type": "object",
        "properties": {
            "model_id": {"title": "The model to use for the inference.", "type": "str"},
            "messages": {
                "title": "The messages for this chat session.",
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "role": {
                            "title": "The role of the message author.",
                            "type": "string",
                            "enum": ["user", "assistant"],
                        },
                        "content": {
                            "title": "The contents of the message.",
                            "type": "string",
                        },
                    },
                    "required": ["role", "content"],
                },
            },
        },
        "required": ["model_id", "messages"],
    }
}

response_schema = {
    "application/json": {
        "oneOf": [
            {
                "$schema": "http://json-schema.org/draft-07/schema#",
                "type": "object",
                "description": "AI Service response for /ai_service_stream",
                "properties": {
                    "choices": {
                        "description": "A list of chat completion choices.",
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "index": {
                                    "type": "integer",
                                    "title": "The index of this result.",
                                },
                                "delta": {
                                    "description": "A message result.",
                                    "type": "object",
                                    "properties": {
                                        "content": {
                                            "description": "The contents of the message.",
                                            "type": "string",
                                        },
                                        "role": {
                                            "description": "The role of the author of this message.",
                                            "type": "string",
                                        },
                                    },
                                    "required": ["role"],
                                },
                            },
                        },
                    }
                },
                "required": ["choices"],
            },
            {
                "$schema": "http://json-schema.org/draft-07/schema#",
                "type": "object",
                "description": "AI Service response for /ai_service",
                "properties": {
                    "choices": {
                        "description": "A list of chat completion choices",
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "index": {
                                    "type": "integer",
                                    "description": "The index of this result.",
                                },
                                "message": {
                                    "description": "A message result.",
                                    "type": "object",
                                    "properties": {
                                        "role": {
                                            "description": "The role of the author of this message.",
                                            "type": "string",
                                        },
                                        "content": {
                                            "title": "Message content.",
                                            "type": "string",
                                        },
                                    },
                                    "required": ["role"],
                                },
                            },
                        },
                    }
                },
                "required": ["choices"],
            },
        ]
    }
}

### Test AI Service locally
Before creating deployment, we can test locally the prepared AI service to detect potential errors at an early stage.

In [26]:
from ibm_watsonx_ai.deployments import RuntimeContext

context = RuntimeContext(api_client=client)

local_function, local_function_stream = gen_ai_service(context)

In [27]:
inference_model_id = client.foundation_models.ChatModels.LLAMA_3_3_70B_INSTRUCT
question = "Summarize the document"

messages = [{"role": "user", "content": question}]

context = RuntimeContext(
    api_client=client,
    request_payload_json={"model_id": inference_model_id, "messages": messages},
)

response = local_function(context)
print(response)

{'headers': {'Content-Type': 'application/json'}, 'body': {'id': 'chatcmpl-5d2e688d-b97a-4feb-b601-de1465177b8b', 'object': 'chat.completion', 'model_id': 'meta-llama/llama-3-3-70b-instruct', 'model': 'meta-llama/llama-3-3-70b-instruct', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "The document is a transcript of a State of the Union address. The speaker, likely the President of the United States, reports that the state of the union is strong because of the strength and resilience of the American people. The speaker highlights the progress made in the past year and expresses confidence that the country will continue to grow stronger.\n\nThe speaker then outlines a Unity Agenda for the Nation, which includes four key areas of focus:\n\n1. Beating the opioid epidemic by increasing funding for prevention, treatment, and recovery, and working to stop the flow of illicit drugs.\n2. Addressing mental health, particularly among children, by strengthening privacy prote

### Create deployment
After making sure that AI service works as expected, we can proceed to the deployment creation step. 

Retrieve the software specification used by the AI service.

In [28]:
software_spec_id = client.software_specifications.get_id_by_name("genai-A25-py3.12")

Create the AI service asset.

In [29]:
ai_service_metadata = {
    client.repository.AIServiceMetaNames.NAME: vector_index_name,
    client.repository.AIServiceMetaNames.DESCRIPTION: "",
    client.repository.AIServiceMetaNames.SOFTWARE_SPEC_ID: software_spec_id,
    client.repository.AIServiceMetaNames.CUSTOM: {},
    client.repository.AIServiceMetaNames.REQUEST_DOCUMENTATION: request_schema,
    client.repository.AIServiceMetaNames.RESPONSE_DOCUMENTATION: response_schema,
}

ai_service_details = client.repository.store_ai_service(
    meta_props=ai_service_metadata, ai_service=gen_ai_service
)
ai_service_id = client.repository.get_ai_service_id(ai_service_details)

Deploy the AI service.

In [30]:
deployment_metadata = {
    client.deployments.ConfigurationMetaNames.NAME: vector_index_name,
    client.deployments.ConfigurationMetaNames.ONLINE: {},
    client.deployments.ConfigurationMetaNames.CUSTOM: {},
    client.deployments.ConfigurationMetaNames.DESCRIPTION: f"{vector_index_name} description",
}

function_deployment_details = client.deployments.create(
    ai_service_id, meta_props=deployment_metadata, space_id=space_id
)
deployment_id = client.deployments.get_id(function_deployment_details)



######################################################################################

Synchronous deployment creation for id: '019e4074-bb82-7414-bbb0-7ffaa1ea9cb1' started

######################################################################################


initializing


Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
........
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='019e4074-d445-775c-b9d0-1d474c2f8735'
-----------------------------------------------------------------------------------------------




Evaluate the deployment of the AI service.

In [31]:
payload = {
    "model_id": inference_model_id,
    "messages": [{"role": "user", "content": question}],
}

result = client.deployments.run_ai_service(deployment_id, payload)
result

{'id': 'chatcmpl-695fe4bb-bbe9-4c2f-a206-f655468d59cd',
 'object': 'chat.completion',
 'model_id': 'meta-llama/llama-3-3-70b-instruct',
 'model': 'meta-llama/llama-3-3-70b-instruct',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "The document is a transcript of a State of the Union address. The speaker, likely the President of the United States, reports that the state of the union is strong because of the strength and resilience of the American people. The speaker highlights the progress made in the past year and expresses confidence that the country will continue to grow stronger.\n\nThe speaker then outlines a Unity Agenda for the Nation, which includes four key areas of focus:\n\n1. Beating the opioid epidemic by increasing funding for prevention, treatment, and recovery, and working to stop the flow of illicit drugs.\n2. Addressing mental health, particularly among children, by strengthening privacy protections, banning targeted advertising to childre

In [32]:
import json

stream_results = client.deployments.run_ai_service_stream(deployment_id, payload)

for chunk in map(json.loads, stream_results):
    if chunk["choices"]:
        print(chunk["choices"][0]["delta"].get("content", ""), flush=True, end="")

The document is a transcript of a State of the Union address. The speaker, likely the President of the United States, reports that the state of the union is strong because of the strength and resilience of the American people. The speaker highlights the progress made in the past year and expresses confidence that the country will continue to grow stronger.

The speaker then outlines a Unity Agenda for the Nation, which includes four key areas of focus:

1. Beating the opioid epidemic by increasing funding for prevention, treatment, and recovery, and working to stop the flow of illicit drugs.
2. Addressing mental health, particularly among children, by strengthening privacy protections, banning targeted advertising to children, and increasing access to mental health services.
3. Supporting veterans by fulfilling the country's sacred obligation to care for them and their families.
4. (Implicitly) Fighting inflation by implementing a plan to lower costs, increase domestic production, and 

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to reproduce the behaviour of chat with a document and vector indexes programmatically through watsonx APIs and clients.
 
Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Mateusz Szewczyk**, Software Engineer at watsonx.ai.

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.